# Building the Prediction Inputs

To predict how much electricity a home uses in a given hour, we give the model two clues: what the home used one hour earlier, 
and what it used at the same hour yesterday. 

## Project folder

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction')

## Loading the hourly sample

In [ ]:
import pandas as pd
hourly = pd.read_csv(PROJECT_ROOT / "data/pilot/hourly_sample.csv")
hourly.head()

,LCLid,hour_start,kWh,n_valid,is_complete
0,MAC000414,2013-01-01 00:00:00+00:00,0.273,2,True
1,MAC000414,2013-01-01 01:00:00+00:00,0.193,2,True
2,MAC000414,2013-01-01 02:00:00+00:00,0.090,2,True
3,MAC000414,2013-01-01 03:00:00+00:00,0.091,2,True
4,MAC000414,2013-01-01 04:00:00+00:00,0.127,2,True


## Keeping only the columns we need

In [ ]:
hourly = hourly[["LCLid", "hour_start", "kWh"]]
hourly.columns = ["LCLid", "hour", "kwh"]
hourly.head()

,LCLid,hour,kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273
1,MAC000414,2013-01-01 01:00:00+00:00,0.193
2,MAC000414,2013-01-01 02:00:00+00:00,0.090
3,MAC000414,2013-01-01 03:00:00+00:00,0.091
4,MAC000414,2013-01-01 04:00:00+00:00,0.127


## Turning the hour into a real timestamp

In [ ]:
hourly["hour"] = pd.to_datetime(hourly["hour"], utc=True)
hourly.head()

,LCLid,hour,kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273
1,MAC000414,2013-01-01 01:00:00+00:00,0.193
2,MAC000414,2013-01-01 02:00:00+00:00,0.090
3,MAC000414,2013-01-01 03:00:00+00:00,0.091
4,MAC000414,2013-01-01 04:00:00+00:00,0.127


## Listing every hour of the year

We list every hour between the first and last reading so we can spot the hours that have no reading.

In [ ]:
hours = pd.date_range(hourly["hour"].min(), hourly["hour"].max(), freq="h")
hours[:5]

DatetimeIndex(['2013-01-01 00:00:00+00:00', '2013-01-01 01:00:00+00:00',
               '2013-01-01 02:00:00+00:00', '2013-01-01 03:00:00+00:00',
               '2013-01-01 04:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq='h')

## One row for every home and every hour

This gives us a full table where missing hours show up as empty instead of just disappearing.

In [ ]:
homes = sorted(hourly["LCLid"].unique())
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "target_hour"])
len(grid)

35040

## Lining up the readings by home and hour

In [ ]:
values = hourly.set_index(["LCLid", "hour"])["kwh"]
values.index.names = ["LCLid", "target_hour"]
values.head()

LCLid      target_hour              
MAC000414  2013-01-01 00:00:00+00:00    0.273
           2013-01-01 01:00:00+00:00    0.193
           2013-01-01 02:00:00+00:00    0.090
           2013-01-01 03:00:00+00:00    0.091
           2013-01-01 04:00:00+00:00    0.127
Name: kwh, dtype: float64

## Filling in the hours with no reading

Missing hours stay empty (NaN). We don't fill them with zero, because zero is a real reading.

In [ ]:
values = values.reindex(grid)
values.head()

LCLid      target_hour              
MAC000414  2013-01-01 00:00:00+00:00    0.273
           2013-01-01 01:00:00+00:00    0.193
           2013-01-01 02:00:00+00:00    0.090
           2013-01-01 03:00:00+00:00    0.091
           2013-01-01 04:00:00+00:00    0.127
Name: kwh, dtype: float64

## Adding the value we want to predict

In [ ]:
all_rows = values.reset_index(name="target_kwh")
all_rows.head()

,LCLid,target_hour,target_kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273
1,MAC000414,2013-01-01 01:00:00+00:00,0.193
2,MAC000414,2013-01-01 02:00:00+00:00,0.090
3,MAC000414,2013-01-01 03:00:00+00:00,0.091
4,MAC000414,2013-01-01 04:00:00+00:00,0.127


## Find the previous hour

In [ ]:
previous_time = all_rows["target_hour"] - pd.Timedelta(hours=1)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], previous_time])
previous_values = values.reindex(keys)
all_rows["lag_1h_kwh"] = previous_values.values
all_rows.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273,NaN
1,MAC000414,2013-01-01 01:00:00+00:00,0.193,0.273
2,MAC000414,2013-01-01 02:00:00+00:00,0.090,0.193
3,MAC000414,2013-01-01 03:00:00+00:00,0.091,0.090
4,MAC000414,2013-01-01 04:00:00+00:00,0.127,0.091


## Find the same hour yesterday

In [ ]:
yesterday_time = all_rows["target_hour"] - pd.Timedelta(hours=24)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], yesterday_time])
yesterday_values = values.reindex(keys)
all_rows["lag_24h_kwh"] = yesterday_values.values
all_rows.iloc[22:27]

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh
22,MAC000414,2013-01-01 22:00:00+00:00,0.238,0.339,NaN
23,MAC000414,2013-01-01 23:00:00+00:00,0.090,0.238,NaN
24,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273
25,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193
26,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090


## Adding the hour of day and weekday

In [ ]:
all_rows["hour"] = all_rows["target_hour"].dt.hour
all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek
all_rows.iloc[22:27]

C:\Users\Mystery\AppData\Local\Temp\ipykernel_70520\3353378970.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek


,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
22,MAC000414,2013-01-01 22:00:00+00:00,0.238,0.339,NaN,22,1
23,MAC000414,2013-01-01 23:00:00+00:00,0.090,0.238,NaN,23,1
24,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273,0,2
25,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193,1,2
26,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090,2,2


## Find rows with missing values

In [ ]:
all_rows["row_status"] = "kept"
all_rows.loc[all_rows["lag_24h_kwh"].isna(), "row_status"] = "missing_lag_24h"
all_rows.loc[all_rows["lag_1h_kwh"].isna(), "row_status"] = "missing_lag_1h"
all_rows.loc[all_rows["target_kwh"].isna(), "row_status"] = "missing_target"
all_rows["row_status"].value_counts()

row_status
kept               34173
missing_lag_24h      422
missing_target       420
missing_lag_1h        25
Name: count, dtype: int64

## Keeping the rows we can use

In [ ]:
features = all_rows[all_rows["row_status"] == "kept"]
features = features.drop(columns="row_status")
features.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
24,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273,0,2
25,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193,1,2
26,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090,2,2
27,MAC000414,2013-01-02 03:00:00+00:00,0.116,0.089,0.091,3,2
28,MAC000414,2013-01-02 04:00:00+00:00,0.108,0.116,0.127,4,2


## How many rows did we keep?

In [ ]:
len(features)

34173

## Counting the gaps for each home

In [ ]:
counts = pd.crosstab(all_rows["LCLid"], all_rows["row_status"])
counts = counts.reindex(columns=["kept", "missing_target",
    "missing_lag_1h", "missing_lag_24h"], fill_value=0)
counts.columns = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
counts["grid_rows"] = all_rows.groupby("LCLid").size()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows
LCLid,,,,,
MAC000414,8534,100,8,118,8760
MAC000675,8727,4,3,26,8760
MAC001312,8735,1,1,23,8760
MAC005456,8177,315,13,255,8760


## Breaking the gaps down by cause

One row can be missing more than one thing, so we also count each one on its own.

In [ ]:
for field, name in [("target_kwh", "any_target_missing"),
                    ("lag_1h_kwh", "any_lag_1h_missing"), ("lag_24h_kwh", "any_lag_24h_missing")]:
    counts[name] = all_rows[field].isna().groupby(all_rows["LCLid"]).sum()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows,any_target_missing,any_lag_1h_missing,any_lag_24h_missing
LCLid,,,,,,,,
MAC000414,8534,100,8,118,8760,100,100,123
MAC000675,8727,4,3,26,8760,4,4,27
MAC001312,8735,1,1,23,8760,1,1,24
MAC005456,8177,315,13,255,8760,315,315,338


## How many targets do we have on day one?

On the first day there is no "yesterday" yet, so these rows cannot be kept. This explains part of the exclusions.

In [ ]:
first_hour = all_rows.groupby("LCLid")["target_hour"].transform("min")
first_day = all_rows["target_hour"] < first_hour + pd.Timedelta(hours=24)
observed = first_day & all_rows["target_kwh"].notna()
counts["target_observed_in_first_24h"] = observed.groupby(all_rows["LCLid"]).sum()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows,any_target_missing,any_lag_1h_missing,any_lag_24h_missing,target_observed_in_first_24h
LCLid,,,,,,,,,
MAC000414,8534,100,8,118,8760,100,100,123,24
MAC000675,8727,4,3,26,8760,4,4,27,24
MAC001312,8735,1,1,23,8760,1,1,24,24
MAC005456,8177,315,13,255,8760,315,315,338,24


## Putting the count table together

In [ ]:
column_order = ["grid_rows", "rows_kept", "excluded_missing_target", "excluded_missing_lag_1h",
                "excluded_missing_lag_24h", "any_target_missing", "any_lag_1h_missing",
                "any_lag_24h_missing", "target_observed_in_first_24h"]
counts = counts[column_order].copy()
counts.loc["TOTAL"] = counts.sum()
counts = counts.reset_index()
counts

,LCLid,grid_rows,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,any_target_missing,any_lag_1h_missing,any_lag_24h_missing,target_observed_in_first_24h
0,MAC000414,8760,8534,100,8,118,100,100,123,24
1,MAC000675,8760,8727,4,3,26,4,4,27,24
2,MAC001312,8760,8735,1,1,23,1,1,24,24
3,MAC005456,8760,8177,315,13,255,315,315,338,24
4,TOTAL,35040,34173,420,25,422,420,420,512,96


## Choosing where to save

In [ ]:
from pathlib import Path
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed')

## Saving the prediction inputs

In [ ]:
feature_path = output_dir / "prediction_inputs_sample.csv"
features.to_csv(feature_path, index=False)
feature_path

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_inputs_sample.csv')

## Saving the counts

In [ ]:
counts_path = output_dir / "prediction_input_counts_sample.csv"
counts.to_csv(counts_path, index=False)
counts_path

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_input_counts_sample.csv')

# Week 2: All development homes

same steps on M2 full hourly file, but only for the homes marked DEVELOPMENT.

## Load full hourly file

In [ ]:
full = pd.read_csv(PROJECT_ROOT / "data/processed/hourly_readings.csv")
full = full[["LCLid", "hour_start", "kWh"]]
full.columns = ["LCLid", "hour", "kwh"]
full["hour"] = pd.to_datetime(full["hour"], utc=True)
print(full["LCLid"].nunique(), "homes,", len(full), "rows")

300 homes, 2628000 rows


## Keeping development homes

The reserved homes are kept aside for later, so we remove them here.

In [ ]:
split = pd.read_csv(PROJECT_ROOT / "data/household_split.csv")
dev_homes = split.loc[split["Cohort"] == "DEVELOPMENT", "LCLid"]
reserved_homes = split.loc[split["Cohort"] != "DEVELOPMENT", "LCLid"]
full = full[full["LCLid"].isin(dev_homes)]
print(len(dev_homes), "development homes,", full["LCLid"].nunique(), "found in the file")

240 development homes, 240 found in the file


## One row for every home and every hour

In [ ]:
hours = pd.date_range(full["hour"].min(), full["hour"].max(), freq="h")
homes = sorted(full["LCLid"].unique())
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "target_hour"])
values = full.set_index(["LCLid", "hour"])["kwh"]
values.index.names = ["LCLid", "target_hour"]
values = values.reindex(grid)
all_rows = values.reset_index(name="target_kwh")
len(all_rows)

2102400

## Previous hour and the same hour yesterday

We look up the exact time 1 hour and 24 hours earlier for the same home. If there is no reading, it stays empty.

In [ ]:
for lag, name in [(1, "lag_1h_kwh"), (24, "lag_24h_kwh")]:
    earlier = all_rows["target_hour"] - pd.Timedelta(hours=lag)
    keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], earlier])
    all_rows[name] = values.reindex(keys).values
all_rows.iloc[22:27]

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh
22,MAC000038,2013-01-01 22:00:00+00:00,0.740,0.947,NaN
23,MAC000038,2013-01-01 23:00:00+00:00,0.645,0.740,NaN
24,MAC000038,2013-01-02 00:00:00+00:00,0.227,0.645,1.188
25,MAC000038,2013-01-02 01:00:00+00:00,0.111,0.227,0.503
26,MAC000038,2013-01-02 02:00:00+00:00,0.136,0.111,0.164


## Adding the date period

TRAIN is Jan-Aug, VALIDATION is Sep-Oct and TEST is Nov-Dec (from data/split_rules.json).

In [ ]:
import json
rules = json.load(open(PROJECT_ROOT / "data/split_rules.json"))
all_rows["period"] = None
for period, (start, end) in rules["periods"].items():
    start = pd.Timestamp(start, tz="UTC")
    end = pd.Timestamp(end, tz="UTC")
    inside = (all_rows["target_hour"] >= start) & (all_rows["target_hour"] < end)
    all_rows.loc[inside, "period"] = period
all_rows["period"].value_counts()

period
TRAIN         1399680
VALIDATION     351360
TEST           351360
Name: count, dtype: int64

## Rows with missing values

In [ ]:
all_rows["row_status"] = "kept"
all_rows.loc[all_rows["lag_24h_kwh"].isna(), "row_status"] = "missing_lag_24h"
all_rows.loc[all_rows["lag_1h_kwh"].isna(), "row_status"] = "missing_lag_1h"
all_rows.loc[all_rows["target_kwh"].isna(), "row_status"] = "missing_target"
all_rows["row_status"].value_counts()

row_status
kept               2084732
missing_target        9809
missing_lag_24h       7146
missing_lag_1h         713
Name: count, dtype: int64

## Keeping Rows we can use

In [ ]:
inputs = all_rows[all_rows["row_status"] == "kept"]
inputs = inputs[["LCLid", "target_hour", "target_kwh", "lag_1h_kwh", "lag_24h_kwh", "period"]]
inputs.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,period
24,MAC000038,2013-01-02 00:00:00+00:00,0.227,0.645,1.188,TRAIN
25,MAC000038,2013-01-02 01:00:00+00:00,0.111,0.227,0.503,TRAIN
26,MAC000038,2013-01-02 02:00:00+00:00,0.136,0.111,0.164,TRAIN
27,MAC000038,2013-01-02 03:00:00+00:00,0.131,0.136,0.143,TRAIN
28,MAC000038,2013-01-02 04:00:00+00:00,0.144,0.131,0.128,TRAIN


## Counts for each period

In [ ]:
period_counts = pd.crosstab(all_rows["period"], all_rows["row_status"])
period_counts = period_counts.reindex(index=["TRAIN", "VALIDATION", "TEST"],
    columns=["kept", "missing_target", "missing_lag_1h", "missing_lag_24h"], fill_value=0)
period_counts.columns = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
period_counts.insert(0, "grid_rows", all_rows.groupby("period").size())
period_counts.loc["TOTAL"] = period_counts.sum()
period_counts = period_counts.reset_index().rename(columns={"index": "period"})
period_counts

,period,grid_rows,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h
0,TRAIN,1399680,1392091,782,547,6260
1,VALIDATION,351360,349127,1806,75,352
2,TEST,351360,343514,7221,91,534
3,TOTAL,2102400,2084732,9809,713,7146


## Checking the result

In [ ]:
no_gaps = inputs[["target_kwh", "lag_1h_kwh", "lag_24h_kwh"]].notna().all().all()
no_reserved = not inputs["LCLid"].isin(reserved_homes).any()
total = period_counts.iloc[-1]
excluded = total.filter(like="excluded").sum()
adds_up = total["rows_kept"] + excluded == total["grid_rows"]
print("No missing target or input:", no_gaps)
print("No reserved homes:", no_reserved)
print("Kept + excluded = grid rows:", adds_up)

No missing target or input: True
No reserved homes: True
Kept + excluded = grid rows: True


## Five sample

We pick five saved rows and look up the two earlier values again from the hourly file.

In [ ]:
raw = pd.read_csv(PROJECT_ROOT / "data/processed/hourly_readings.csv")
raw["hour_start"] = pd.to_datetime(raw["hour_start"], utc=True)
raw = raw.set_index(["LCLid", "hour_start"])["kWh"]
sample = inputs.sample(5, random_state=42).copy()
sample["file_lag_1h"] = [raw[(h, t - pd.Timedelta(hours=1))] for h, t in zip(sample["LCLid"], sample["target_hour"])]
sample["file_lag_24h"] = [raw[(h, t - pd.Timedelta(hours=24))] for h, t in zip(sample["LCLid"], sample["target_hour"])]
sample["match"] = (sample["file_lag_1h"] == sample["lag_1h_kwh"]) & (sample["file_lag_24h"] == sample["lag_24h_kwh"])
sample

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,period,file_lag_1h,file_lag_24h,match
299924,MAC000855,2013-03-28 20:00:00+00:00,1.171,0.945,2.322,TRAIN,0.945,2.322,True
1456643,MAC003955,2013-04-14 11:00:00+00:00,0.047,0.049,0.071,TRAIN,0.049,0.071,True
1622166,MAC004228,2013-03-07 06:00:00+00:00,0.112,0.126,0.190,TRAIN,0.126,0.190,True
248132,MAC000744,2013-04-29 20:00:00+00:00,0.734,1.473,0.253,TRAIN,1.473,0.253,True
545130,MAC001659,2013-03-25 18:00:00+00:00,1.288,0.292,0.324,TRAIN,0.292,0.324,True


#### Save the prediction inputs and counts

In [ ]:
inputs_path = output_dir / "prediction_inputs.csv"
inputs.to_csv(inputs_path, index=False)
counts_path = output_dir / "prediction_input_counts.csv"
period_counts.to_csv(counts_path, index=False)
inputs_path, counts_path

(WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_inputs.csv'),
 WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_input_counts.csv'))

Can M4 open them?

In [ ]:
check = pd.read_csv(inputs_path)
print(check.shape)
print(pd.read_csv(counts_path))

(2084732, 6)
       period  grid_rows  rows_kept  excluded_missing_target  \
0       TRAIN    1399680    1392091                      782   
1  VALIDATION     351360     349127                     1806   
2        TEST     351360     343514                     7221   
3       TOTAL    2102400    2084732                     9809   

   excluded_missing_lag_1h  excluded_missing_lag_24h  
0                      547                      6260  
1                       75                       352  
2                       91                       534  
3                      713                      7146  
